# CSCI 447/547 Hackathon 3: Logistic Regression/Regularization

This notebook is designed to be started during class and continued as a take-home activity

## How to use hackathon notebooks:

If the topic covered in a hackathon is new to you, work through the cells in order and read the explanation before running each code cell. You do not need to understand every detail of the hackathon on the first pass, instead, focus on the approach we take:

1. **Look at the data**
2. **Separate inputs from the target we want to predict**
3. **Split the data so we can test whether the model generalizes**
4. **Fit a model using the training data**
5. **Make predictions and evaluate them**
6. **Improve the model carefully <u>without</u> using the final evaluation data to make decisions**

We will work through the salary example together as a class. Pause before important code cells and think about what you expect to see. Please ask Lucy or a TA questions any time a term or line of code is unfamiliar.

After class, continue from wherever you left off. The existing explanations and code will be there to guide you through the process. Complete the marked answer sections, run every cell, and explain what the results mean in language that makes sense for you. Hackathons will not be graded, they are only to help you, and you will get out of them what you put into them.

<h4><span style="color:red">The goal of this notebook is NOT to memorize every function. It is to identify the processes we use in machine learning and be able to reuse them.</span></h4>

##### In this exercise, you will implement logistic regression and see how regularization affects your results.

---

### Google Colab Instructions

If you are using Google Colab <a href="https://colab.research.google.com/github/lucywowen/csci547_ML/blob/main/examples/Linear_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Google Colab"/></a> you will need to download [these two files](https://github.com/lucywowen/csci547_ML/tree/main/examples/data/logistic_regression_hack) and upload them to Colab for this to work.

---

### Local (VSCode/Jupyter Lab) Version

Make sure you

```bash
git pull
```

for the latest version of the repository and make sure that your `uv` virtual environment is enabled.

---

## 1. Logistic Regression

In Hackathon 1, we used Linear Regression to predict a continuous, unbounded value (profit). Today, we transition to classification.

**The Scenario:** Suppose you are the administrator of a university department. You want to determine each applicant’s chance of admission based on their results on two exams. You have historical data from previous applicants that you can use as a training set. 

The file `ex2data1.txt` contains our dataset:
* Column 1: Exam 1 score
* Column 2: Exam 2 score
* Column 3: Admissions decision (0 = Not Admitted, 1 = Admitted)

### Think-Pair-Share 1: Why not Linear Regression?
**Context:** Our target variable $y$ is now strictly binary $\{0, 1\}$. 

1. **Think:** From a mathematical and logical perspective, why is standard Linear Regression ($h_\theta(x) = \theta^Tx$) inappropriate for this classification task? What would happen if we tried to fit a straight line to binary outputs?

    *Your individual hypothesis:* **ANSWER**

2. **Pair:** Discuss with your group. 

    *Group consensus:* **ANSWER**

3. **Share:** Be prepared to share your group's conclusion with the class.

---

### 1.1 Visualizing the Data

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
%matplotlib inline

In [ ]:
try:
    from google.colab import files # Colab file import
    uploaded = files.upload()
    
    df = pd.read_csv('ex2data1.txt', sep=',', header=None) # colab file import
    df.columns = ['exam_score_1', 'exam_score_2', 'label']
    
except: 
    
    df = pd.read_csv('data/logistic_regression_hack/ex2data1.txt', sep=',', header=None) # local file import
    df.columns = ['exam_score_1', 'exam_score_2', 'label']


In [ ]:
df.describe().T

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.scatterplot(x='exam_score_1', y='exam_score_2', hue='label', data=df, style='label', s=80)
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[1:], ['Not admitted', 'Admitted'])
plt.title('Scatter plot of training data')
plt.show(ax)

### 1.2 The Mathematics of Classification

To adapt our linear model for classification, we need to bound our predictions between $0$ and $1$ so they can be interpreted as probabilities. Let's map the formal math to our code:

#### 1.2.1 The Sigmoid (Logistic) Function:
 
We pass our linear equation $\theta^Tx$ through the sigmoid function $g(z)$. This "squashes" any real number into the $(0, 1)$ range.
$$h_\theta(x) = g(\theta^Tx) = \frac{1}{1+e^{-\theta^Tx}}$$

In [ ]:
def sigmoid(z):
    z = np.array(z)
    return 1 / (1+np.exp(-z))

Plot of sigmoid function:

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
z = np.linspace(-10, 10, 100)
sig = sigmoid(z)
plt.figure(figsize=(9, 3))
plt.plot([-10, 10], [0, 0], "k-")
plt.plot([-10, 10], [0.5, 0.5], "k:")
plt.plot([-10, 10], [1, 1], "k:")
plt.plot([0, 0], [-1.1, 1.1], "k-")
plt.plot(z, sig, "b-", linewidth=2)
plt.xlabel("z")
plt.axis([-10, 10, -0.1, 1.1])
plt.show()

#### 1.2.2 The Cost Function (Cross-Entropy / Log Loss): 
In Linear Regression, we used Mean Squared Error (MSE). However, passing MSE through a sigmoid function creates a "non-convex" cost surface full of local minima, meaning gradient descent might get stuck. Instead, we use **Log Loss**, which heavily penalizes confident but incorrect predictions:
$$J(\theta) = -\frac{1}{m}\sum_{i=1}^m[y^{(i)} \log(h_\theta(x^{(i)}))+(1-y^{(i)})\log(1-h_\theta(x^{(i)}))]$$

*Vectorized implementation:*
$$J(\theta) = \frac{1}{m}(-y^T \log(h)-(1-y)^T\log(1-h))$$

In [ ]:
def cost_function(theta, X, y):
    m = y.shape[0]
    theta = theta.reshape(-1, 1)
    h = sigmoid(X.dot(theta))
    
    J = (1/m) * (-y.T.dot(np.log(h)) - (1-y).T.dot(np.log(1-h)))

    diff_hy = h - y
    grad = (1/m) * X.T.dot(diff_hy)

    return J.flatten()[0], grad.flatten()

In [ ]:
m = df.shape[0]
X = np.hstack((np.ones((m,1)),df[['exam_score_1', 'exam_score_2']].values))
y = np.array(df.label.values).reshape(-1,1)
initial_theta = np.zeros(shape=(X.shape[1]))

#### 1.2.3 The Gradient: 
Interestingly, the calculus derivative of the log loss function results in an update rule that looks computationally identical to the linear regression gradient.
$$\nabla J(\theta) = \frac{1}{m} \cdot X^T \cdot (g(X\theta)-y)$$

In [ ]:
cost, grad = cost_function(initial_theta, X, y)
print('Cost at initial theta (zeros):', cost)
print('Expected cost (approx): 0.693')
print('Gradient at initial theta (zeros):')
print(grad.T)
print('Expected gradients (approx):\n -0.1000\n -12.0092\n -11.2628')

In [ ]:
test_theta = np.array([-24, 0.2, 0.2])
[cost, grad] = cost_function(test_theta, X, y)

print('Cost at test theta:', cost)
print('Expected cost (approx): 0.218')
print('Gradient at test theta:')
print(grad.T)
print('Expected gradients (approx):\n 0.043\n 2.566\n 2.647')

### 1.2.4 Learning Parameters with Advanced Optimizers

In Hackathon 1, we wrote a `for` loop to manually compute gradient descent, requiring us to carefully guess the learning rate $\alpha$. Here, we will use `scipy.optimize.minimize` with the **TNC (Truncated Newton Algorithm)** method.

### Think-Pair-Share 2: Beyond Vanilla Gradient Descent
**Context:** Advanced optimizers like TNC or BFGS do not require us to manually set a learning rate $\alpha$. Instead, they often compute or approximate the second derivative of the cost function (the "Hessian").

1. **Think:** Why might an algorithm that uses the second derivative be faster and more robust than our manual gradient descent loop that only uses the first derivative?

    *Your individual hypothesis:* **ANSWER**

1. **Pair:** Discuss with your group. 

    *Group consensus:* **ANSWER**

1. **Share:** Be prepared to share your group's conclusion with the class.

In [ ]:
import scipy.optimize as opt
def optimize_theta(X, y, initial_theta):
    opt_results = opt.minimize(cost_function, initial_theta, args=(X, y), method='TNC',
                               jac=True, options={'maxfun':400})
    return opt_results['x'], opt_results['fun']

In [ ]:
opt_theta, cost = optimize_theta(X, y, initial_theta)

In [ ]:
print('Cost at theta found by fminunc:', cost)
print('Expected cost (approx): 0.203')
print('theta:\n', opt_theta.reshape(-1,1))
print('Expected theta (approx):')
print(' -25.161\n 0.206\n 0.201')

#### 1.2.5 Decision Boundary

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.scatterplot(x='exam_score_1', y='exam_score_2', hue='label', data=df, style='label', s=80)
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[1:], ['Not admitted', 'Admitted'])
plt.title('Training data with decision boundary')

plot_x = np.array(ax.get_xlim())
plot_y = (-1/opt_theta[2]*(opt_theta[1]*plot_x + opt_theta[0]))
plt.plot(plot_x, plot_y, '-', c="green")
plt.show(ax)

#### 1.2.6 Evaluating Logistic Regression

Predict whether a particular student will be admitted

In [ ]:
prob = sigmoid(np.array([1, 45, 85]).dot(opt_theta))
print('For a student with scores 45 and 85, we predict an admission probability of', prob)
print('Expected value: 0.775 +/- 0.002');

Accuracy on training set

In [ ]:
def predict(X, theta):
    y_pred = [1 if sigmoid(X[i, :].dot(theta)) >= 0.5 else 0 for i in range(0, X.shape[0])]
    return y_pred

In [ ]:
X = np.hstack((np.ones((m,1)),df[['exam_score_1', 'exam_score_2']].values))

y_pred_prob = predict(X, opt_theta)
f'Train accuracy: {np.mean(y_pred_prob == df.label.values) * 100}'

#### 1.2.7 Equivalent code using Scikit-Learn:

In [ ]:
from sklearn.linear_model import LogisticRegression
log_reg = LogisticRegression(solver='newton-cg', max_iter=400)
log_reg.fit(df[['exam_score_1', 'exam_score_2']].values,
            df.label.values)

In [ ]:
log_reg.intercept_, log_reg.coef_

Sklearn logistic regression accuracy:

In [ ]:
log_reg.score(df[['exam_score_1', 'exam_score_2']].values,
              df.label.values)

## 2. Regularized Logistic Regression

In this section, we tackle a dataset that cannot be separated by a simple straight line, forcing us to deal with model complexity and the risk of overfitting.

**The Scenario:** Suppose you are the product manager of a microchip factory. During QA, each microchip goes through two different tests. You want to determine whether microchips should be accepted or rejected based on these two test scores.

* Column 1: Test 1 score
* Column 2: Test 2 score
* Column 3: Chip accepted or rejected (0 = not accepted, 1 = accepted)

The file `ex2data2.txt` contains our dataset.

---

### 2.1 Visualizing the Data

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    
    df2 = pd.read_csv('ex2data2.txt', sep=',', header=None) # colab file import
    df2.columns = ['test_1', 'test_2', 'label']
    
except: 
    
    df2 = pd.read_csv('data/logistic_regression_hack/ex2data2.txt', sep=',', header=None) # local file import
    df2.columns = ['test_1', 'test_2', 'label']


In [ ]:
df2.describe().T

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.scatterplot(x='test_1', y='test_2', hue='label', data=df2, style='label', s=80)
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[1:], ['accepted', 'rejected'])
plt.title('Scatter plot of training data')
plt.show(ax)

### 2.2 Feature Mapping & The Curse of Dimensionality

### Think-Pair-Share 3: Non-Linear Boundaries
**Context:** Look at the scatter plot of the microchip QA data above. 

1. **Think:** Can a straight line (a linear decision boundary) successfully separate the accepted chips from the rejected ones? How could we mathematically manipulate our two features ($x_1, x_2$) to allow a logistic regression model to draw a curve or circle?

    *Your individual hypothesis:* **ANSWER**

1. **Pair:** Discuss with your group.
   
    *Group consensus:* **ANSWER**

1. **Share:** Be prepared to share your group's conclusion with the class.

---
**Implementation:** To fit this data, we will create more features. We will map the features into all polynomial terms of $x_1$ and $x_2$ up to the sixth power. Our vector of 2 features has now been transformed into a **28-dimensional vector**.

In [ ]:
def map_feature(X1, X2, degree):
    X1 = np.array(X1).reshape(-1,1)
    X2 = np.array(X2).reshape(-1,1)

    out = np.ones((X1.shape[0], 1))
    for i in range(1, degree+1):
        for j in range(0, i+1):
            p = (X1**(i-j)) * (X2**j)
            out = np.append(out, p, axis=1)
    return out

In [ ]:
X_p = map_feature(df2.test_1.values, df2.test_2.values, 6)
X_p.shape

### 2.3 Regularization and the Penalty Term

Mapping our data into 28 dimensions gives our model incredible flexibility, but introduces a massive risk of **overfitting** (the model memorizing the training noise rather than the actual pattern). 

To prevent overfitting, we add a **Regularization Penalty ($\lambda$)** to our cost function. This penalizes large weights, mathematically forcing the model to prefer simpler decision boundaries.

**Regularized Cost Function:**
$$J(\theta) = -\frac{1}{m}\sum_{i=1}^m[y^{(i)} \log(h_\theta(x^{(i)}))+(1-y^{(i)})\log(1-h_\theta(x^{(i)}))] + \frac{\lambda}{2m} \sum_{j=1}^n\theta_j^2$$
*(Note: We do not regularize the bias term $\theta_0$)*

### Think-Pair-Share 4: Tuning Lambda ($\lambda$)
**Context:** The regularization parameter $\lambda$ controls how heavily we penalize the model for having large weights.

1. **Think:** What will happen to our decision boundary if $\lambda = 0$? What if we set $\lambda$ to a high number, like $\lambda = 10,000$? 

    *Your individual hypothesis:* **ANSWER**

2. **Pair:** Discuss with your group. Which extreme causes *overfitting* and which causes *underfitting*?

    *Group consensus:* **ANSWER**

1. **Share:** Be prepared to share your group's conclusion with the class.

In [ ]:
def cost_function_reg(theta, X, y, lambda_reg):
    m = y.shape[0]
    theta = theta.reshape(-1, 1)
    h = sigmoid(X.dot(theta))
    
    J = (1/m) * (-y.T.dot(np.log(h)) - (1-y).T.dot(np.log(1-h))) + (lambda_reg/(2*m)) * np.sum(theta[1:]**2)

    diff_hy = h - y
    grad = (1/m) * X.T.dot(diff_hy) + (lambda_reg/m) * theta
    grad[0] = (1/m) * X[:, 0:1].T.dot(diff_hy)

    return J.flatten()[0], grad.flatten()

#### 2.3.1 Learning Parameters

In [ ]:
import scipy.optimize as opt

def optimize_theta_reg(X, y, initial_theta, lambda_reg):
    opt_results = opt.minimize(
        cost_function_reg, 
        initial_theta, 
        args=(X, y, lambda_reg), 
        method='TNC', 
        jac=True, 
        options={'maxfun': 400} 
    )
    return opt_results['x'], opt_results['fun']

In [ ]:
m = df.shape[0]
X = X_p
y = np.array(df2.label.values).reshape(-1,1)
initial_theta = np.zeros(shape=(X.shape[1]))

In [ ]:
lambda_reg = 1
cost, grad = cost_function_reg(initial_theta, X, y, lambda_reg)
print('Cost at initial theta (zeros):', cost)
print('Expected cost (approx): 0.693')
print('Gradient at initial theta (zeros) - top 5:')
print(grad.T[:5])
print('Expected gradients top 5(approx):\n [0.0085, 0.0188, 0.0001, 0.0503, 0.0115]')

In [ ]:
lambda_reg = 10
initial_theta = np.ones(shape=(X.shape[1]))
cost, grad = cost_function_reg(initial_theta, X, y, lambda_reg)
print('Cost at initial theta:', cost)
print('Expected cost (approx): 3.16')
print('Gradient at theta - top 5:')
print(grad.T[:5])
print('Expected gradients top 5(approx):\n 0.3460\n 0.1614\n 0.1948\n 0.2269\n 0.0922')

### 2.4 Plotting the Decision Boundary

In [ ]:
lambda_reg = [0, 1, 10, 100,]
fig, axs = plt.subplots(nrows=1, ncols=4, figsize=(15,4))
u = np.linspace(-1, 1.5, 50)
v = np.linspace(-1, 1.5, 50)

for il, l in enumerate(lambda_reg):
    theta_opt, cost = optimize_theta_reg(X, y, initial_theta, l)
    z = np.zeros((u.shape[0], v.shape[0]))
    
    for i in range(len(u)):
        for j in range(len(v)):
            z[i,j] = map_feature(u[i], v[j], 6).dot(theta_opt)[0]

    sns.scatterplot(x='test_1', y='test_2', hue='label', data=df2, style='label', s=80, ax=axs[il])

    axs[il].contour(u, v, z.T, levels=[0], colors='green')

    axs[il].set_title(r'$\lambda={}$'.format(l))
    
fig.tight_layout()
plt.show()

### 2.5 Accuracy on Training Set

In [ ]:
lambda_reg = 1
theta, cost = optimize_theta_reg(X, y, initial_theta, lambda_reg)
theta

In [ ]:
y_pred_prob = predict(X, theta)
f'Train accuracy: {np.mean(y_pred_prob == df2.label.values) * 100}'

### 2.6 Equivalent Code using Scikit-Learn

As always, in the real world, we rely on highly optimized libraries. Notice that `LogisticRegression` in `scikit-learn` applies L2 Regularization (the $\lambda$ penalty) *by default*, handled by the hyperparameter `C` (where $C = \dfrac{1}{\lambda}$).

In [ ]:
from sklearn.linear_model import LogisticRegression
log_reg = LogisticRegression(solver='newton-cg', max_iter=400)
log_reg.fit(X[:,1:], df2.label.values)

In [ ]:
log_reg.intercept_, log_reg.coef_

In [ ]:
log_reg.score(X[:,1:], df2.label.values)

---

## Hackathon Takeaways & Synthesis

As your group finishes this notebook, collaborate to answer the following synthesis questions. 

1. **Trade-off Analysis:** 
In Part 2, we mapped 2 features into 28 polynomial features to capture a non-linear boundary. What is the primary computational and mathematical trade-off of artificially expanding your feature space like this?

    **ANSWER**

2. **Diagnosing Failures:** 
You train a Logistic Regression model and achieve 99.9% accuracy on your training data, but it performs terribly on new, unseen generalization data. Based on the $\lambda$ plots generated in section 2.4, how would you adjust your regularization hyperparameter to fix this?

    **ANSWER**


3. **3-2-1 Summary:**
* **3** key mathematical or programming concepts your group solidified today:

    1. 

    2. 

    3. 

* **2** things you are still slightly confused about (we will address these in the next lecture!):

    1. 

    2. 
* **1** real-world classification scenario where a False Positive is much worse than a False Negative:

    1.